# 04 - Parte 2: M2, M3, M4, E1 e E4

Este notebook roda a Parte 2 do projeto, reaproveitando o pipeline da Parte 1 (mesmas divisões, janelas, aumento, treino e métricas). Todo o código fica em `src/birdclef/`; aqui só se chama e se mostra.

| Seção | O que faz | Módulo |
|---|---|---|
| 0 | Ambiente, GPU e checagem dos dados | — |
| 1 | Gera as entradas do M3 (log-Mel 64 × 501, PANNs) e do M4 (embeddings do AST) | `preproc`, `aumento`, `ast_emb` |
| 2 | Confere que todas as entradas têm as **mesmas janelas** | `experimentos_p2.checar_mesmas_janelas` |
| 3 | Modelos M1-M4 (arquitetura e parâmetros) | `modelos` |
| 4 | **E1**: 4 modelos × 5 folds, teste fixo de 600 gravações (H1, H2) | `experimentos_p2.e1` |
| 5 | Comparações pareadas por fold e *bootstrap* no teste | `experimentos_p2.comparacoes_e1` |
| 6 | **E2b + E3b**: sem × com aumento e robustez a ruído em todos os modelos (H3) | `e2b`, `e3b` |
| 7 | **E4**: 25 / 50 / 100% das gravações de treino (H1) | `e4` |
| 8 | Análises: F1 por espécie, vazamento por gravador, t-SNE, Grad-CAM | `analise` |
| 9 | Figuras finais para o relatório | `graficos.todas_parte2` |

**Retomada:** cada treino vira um checkpoint em `checkpoints/` e é pulado se já existir. Se a sessão cair, basta rodar de novo.

**Custo aproximado numa RTX 3050 Laptop (4 GB):** M1/M2 ~10 min por fold; M3 ~40-70 min por fold; geração dos embeddings do AST ~20-30 min; M4 < 1 min por fold.

**Desvios em relação à proposta (registrados no relatório):**
- M4 é o AST **congelado + cabeça linear** (*linear probe*): o ajuste fino completo do AST não cabe em 4 GB. Por isso o M4 não usa aumento de dados.
- E2 nos modelos novos foi reduzido a "sem" × "todas" no fold 0 (1 semente), por custo de GPU; a ablação completa (5 configurações × 3 sementes) fica com o M1 (Parte 1).
- E4 roda em M1, M3 e M4 (fold 0, semente 42).

In [ ]:
# Rode a partir da raiz do repositório (no VS Code: abra a pasta birdclef-dl e selecione o kernel do .venv)
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("pasta:", Path.cwd())

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sem GPU")
for p in ["dados/subconjunto/divisoes.csv", "dados/espectrogramas/mel128_32k/X.npy", "dados/ruido/ruido.csv", "dados/bruto/train_audio"]:
    print(f"{'ok  ' if Path(p).exists() else 'FALTA'} {p}")

## 1. Entradas do M3 e do M4

- **M3 (PANNs CNN14):** log-Mel na configuração original do PANNs (32 kHz, `n_fft` 1024, `hop` 320, 64 bandas, 50-14000 Hz → 64 × 501). Mesmo recorte de janelas do `mel128_32k`; também gera os ruídos de treino (`R.npy`) e o teste ruidoso (E3), com os mesmos ruídos sorteados.
- **M4 (AST):** embedding de 768 dimensões do AST pré-treinado no AudioSet, calculado uma vez por janela a 16 kHz, nos mesmos `inicio_s`.

In [ ]:
import pandas as pd
from birdclef import preproc, aumento, ast_emb, dados

div = pd.read_csv(dados.pasta_dados() / "subconjunto" / "divisoes.csv")
cfg = preproc.MEL64_32K
pasta = preproc.pasta_config(cfg)
if not (pasta / "X.npy").exists():
    preproc.preprocessar(div, cfg)
if not (pasta / "R.npy").exists():
    aumento.preparar_ruido_treino(cfg)
if not (pasta / "janelas_teste.csv").exists():
    aumento.preparar_teste_ruidoso(cfg)
ast_emb.gerar()   # pula o que já existe

## 2. Mesmas janelas em todas as entradas
Se alguma linha vier `False`, as comparações entre modelos não valem: pare e investigue.

In [ ]:
from birdclef import experimentos_p2 as e
chk = e.checar_mesmas_janelas(); display(chk)
assert chk["iguais_ao_mel128_32k"].all()

## 3. Modelos

| ID | Arquitetura | Entrada | Parâmetros treináveis |
|---|---|---|---|
| M1 | CNN 4 blocos (Parte 1) | 128 × 313 | ~393 mil |
| M2 | CRNN: 4 blocos conv + GRU bidirecional (96) | 128 × 313 | ~375 mil (mesmo porte do M1, para H2) |
| M3 | PANNs CNN14, pesos do AudioSet, ajuste fino completo | 64 × 501 | ~80 milhões |
| M4 | cabeça linear sobre o AST congelado | 768 | ~15 mil |

Hiperparâmetros em `experimentos_p2.CFG_TREINO`. O M3 usa lote 16, 15 épocas, taxa 1e-4 no corpo pré-treinado e 1e-3 na cabeça nova; as máscaras do SpecAugment são reescaladas para 64 bandas e quadros de 10 ms.

In [ ]:
from birdclef import modelos
pd.DataFrame([{"modelo": m, "entrada": modelos.ENTRADA[m],
               "parametros": modelos.n_parametros(modelos.criar(m, 20, pretreinado=False)),
               **{k: v for k, v in vars(e.CFG_TREINO[m]).items() if k != "modelo"}} for m in e.MODELOS])

## 4. E1: arquiteturas (H1, H2)

Cada modelo treina 5 vezes (fold *k* como validação, os outros 4 como treino) e cada um dos 5 é avaliado no mesmo teste. Reportamos média ± desvio dos 5 e o *ensemble* (média das probabilidades). O fold 0 do M1 é o `M1_todas_s42_f0` da Parte 1.

Ordem sugerida: modelos baratos primeiro, M3 por último (é o mais longo).

In [ ]:
e1 = e.e1(("M1", "M2", "M4"))
e1 = e.e1(("M3",))

In [ ]:
res = pd.read_csv("resultados/e1/e1_resumo.csv", header=[0, 1], index_col=0)
cols = [c for c in res.columns if c[0] in ("val_f1_macro", "teste_f1_macro", "teste_acc_top1", "teste_acc_top3", "teste_map", "minutos")]
display(res[cols].round(3))
display(pd.read_csv("resultados/e1/e1_ensemble.csv").round(3))

## 5. Comparações (H1, H2)

- **Por fold:** a diferença de F1-macro no teste entre dois modelos treinados no mesmo fold (pareado). `folds_A_melhor` conta em quantos dos 5 o primeiro ganhou.
- **Bootstrap pareado:** reamostra as 600 gravações de teste 5000 vezes, para os dois *ensembles* ao mesmo tempo; dá o intervalo de 95% da diferença. Se o intervalo não cruza zero, a diferença é consistente.

H2 é apoiada se **M2 − M1 > 0**; H1, se **M3 − M1** e **M3 − M2** (e M4 − M1) forem > 0.

In [ ]:
display(e.comparacoes_e1().round(4))

## 6. E2b e E3b: aumento de dados e robustez a ruído (H3)
M2 e M3 também treinam sem aumento no fold 0. Depois todos os modelos (fold 0, semente 42) são testados com ruído da ESC-50 do fold 5, nunca visto no treino.

In [ ]:
e2b = e.e2b(); display(e2b[["modelo", "aumento", "val_f1_macro", "teste_f1_macro"]].round(3))
e3b = e.e3b()
tab = e3b.pivot_table(index=["modelo", "aumento"], columns="teste", values="f1_macro")[["limpo", "20 dB", "10 dB", "5 dB", "0 dB"]]
tab["queda limpo→0 dB"] = tab["0 dB"] - tab["limpo"]
display(tab.round(3))

## 7. E4: eficiência de dados (H1)
Mesmo fold 0, com 25%, 50% e 100% das gravações de treino (estratificado por espécie e aninhado: as 25% estão dentro das 50%). Se o pré-treino ajuda mais com poucos dados, a distância entre M3/M4 e M1 cresce à esquerda.

In [ ]:
e4 = e.e4()
display(e4.pivot_table(index="modelo", columns="fracao", values="teste_f1_macro").round(3))

## 8. Análises
### 8.1 F1 por espécie (ensembles do E1)

In [ ]:
import json, numpy as np
from birdclef import analise
classes = json.loads(open("resultados/e1/classes.json").read())
y = np.load("resultados/e1/y_teste.npy")
done = pd.read_csv("resultados/e1/e1_execucoes.csv")
probs = {m: e.probs_e1(m, sorted(done.loc[done.modelo == m, "fold"])).mean(0) for m in done.modelo.unique()}
nomes = dados.carregar_metadados().drop_duplicates("primary_label").set_index("primary_label")["common_name"]
f1e = analise.f1_por_especie(y, probs, classes); f1e.index = [f"{c} ({nomes.get(c, '')})" for c in f1e.index]
display(f1e.sort_values(list(probs)[0]).round(2))

### 8.2 Vazamento por gravador

As divisões são por gravação, mas o mesmo gravador (`author`) pode ter várias gravações no treino e no teste: mesmo lugar, mesmo microfone, às vezes o mesmo indivíduo. Se o desempenho for bem maior quando o autor também está no treino, parte do acerto vem do contexto da gravação. É uma limitação a registrar no relatório (e uma melhoria futura: `StratifiedGroupKFold` por autor).

In [ ]:
X1, jan1, _ = e.dados_modelo("M1")
arquivos_teste = jan1[jan1.particao == "teste"].drop_duplicates("filename")["filename"]
display(analise.vazamento_autor(pd.Index(arquivos_teste), y, probs).round(3))

### 8.3 t-SNE dos embeddings (teste, M1 × M3, fold 0)

In [ ]:
import matplotlib.pyplot as plt
from birdclef.experimentos import carregar_checkpoint, PASTA_CHECKPOINTS
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, m in zip(axs, ["M1", "M3"]):
    X, jan, _ = e.dados_modelo(m)
    idx = np.flatnonzero((jan.particao == "teste").to_numpy())
    r = carregar_checkpoint(PASTA_CHECKPOINTS / f"{e.nome_execucao(m, 'todas', 42, 0)}.pt", m)
    Z = analise.tsne(analise.embeddings(r.modelo, X, idx, r.norm))
    lab = jan.iloc[idx]["primary_label"].map({c: i for i, c in enumerate(classes)})
    ax.scatter(Z[:, 0], Z[:, 1], c=lab, cmap="tab20", s=3); ax.set_title(f"t-SNE {m} (janelas de teste)"); ax.set_xticks([]); ax.set_yticks([])
fig.savefig("resultados/e1/tsne_M1_M3.png", bbox_inches="tight", dpi=150); plt.show()

### 8.4 Grad-CAM: o modelo olha para o canto?

In [ ]:
import torch
X, jan, _ = e.dados_modelo("M1")
r = carregar_checkpoint(PASTA_CHECKPOINTS / "M1_todas_s42_f0.pt", "M1")
idx = np.flatnonzero((jan.particao == "teste").to_numpy())[::400][:4]
fig, axs = plt.subplots(2, len(idx), figsize=(4 * len(idx), 5))
for j, i in enumerate(idx):
    x = torch.from_numpy(np.asarray(X[i], dtype=np.float32)).to(next(r.modelo.parameters()).device)
    cam = analise.grad_cam(r.modelo, r.modelo.extrator[-1], (x - r.norm[0]) / r.norm[1])
    axs[0, j].imshow(np.asarray(X[i], dtype=np.float32), origin="lower", aspect="auto", cmap="magma"); axs[0, j].set_title(jan.primary_label.iloc[i])
    axs[1, j].imshow(np.asarray(X[i], dtype=np.float32), origin="lower", aspect="auto", cmap="gray"); axs[1, j].imshow(cam, origin="lower", aspect="auto", cmap="jet", alpha=0.45)
fig.savefig("resultados/e1/gradcam_M1.png", bbox_inches="tight", dpi=150); plt.show()

## 9. Figuras para o relatório

In [ ]:
from IPython.display import Image
from birdclef import graficos
graficos.todas_parte2()
for f in ["e1/e1_f1_teste.png", "e1/e1_curvas.png", "e3b/e3b_snr.png", "e4/e4_fracao.png", "e1/matriz_M3.png"]:
    if Path("resultados", f).exists(): display(Image(f"resultados/{f}"))

## 10. Conclusões (preencher com os números acima)

- **H1** (pré-treinados > do zero): ver seções 4, 5 e 7.
- **H2** (CRNN > CNN de mesmo porte): ver `M2 − M1` na seção 5.
- **H3** (aumento reduz a queda sob ruído): ver a coluna `queda limpo→0 dB` na seção 6.
- **Limitações:** M4 sem ajuste fino nem aumento; E2/E3 dos modelos novos com 1 semente; possível vazamento por gravador (seção 8.2); subconjunto equilibrado de 20 espécies (não testa espécies raras).